=============================================================
BOOKLY - Step 1: Exploratory Data Analysis (EDA)
=============================================================
Dataset : books.csv (11,127 rows × 12 columns)
Target  : average_rating (predict rating 0-5)
=============================================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Drop lines with bad format ~8 rows
df_raw = pd.read_csv('../data/books.csv', engine='python', on_bad_lines='skip')
df_raw.columns = df_raw.columns.str.strip()

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"Columns: {df_raw.columns.tolist()}")

Loaded: 11119 rows × 12 columns
Columns: ['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


In [7]:
print("\n" + "="*60)
print("Visualize Data")
print("="*60)
print(df_raw.head())
print()
print(df_raw.dtypes)
print()
print("Nulls per column:")
print(df_raw.isnull().sum())


Visualize Data
   bookID                                              title  \
0       1  Harry Potter and the Half-Blood Prince (Harry ...   
1       2  Harry Potter and the Order of the Phoenix (Har...   
2       4  Harry Potter and the Chamber of Secrets (Harry...   
3       5  Harry Potter and the Prisoner of Azkaban (Harr...   
4       8  Harry Potter Boxed Set  Books 1-5 (Harry Potte...   

                      authors  average_rating        isbn         isbn13  \
0  J.K. Rowling/Mary GrandPré            4.57  0439785960  9780439785969   
1  J.K. Rowling/Mary GrandPré            4.49  0439358078  9780439358071   
2                J.K. Rowling            4.42  0439554896  9780439554893   
3  J.K. Rowling/Mary GrandPré            4.56  043965548X  9780439655484   
4  J.K. Rowling/Mary GrandPré            4.78  0439682584  9780439682589   

  language_code  num_pages  ratings_count  text_reviews_count  \
0           eng        652        2095690               27591   
1           

In [28]:
# 2. Target Variable:  average_rating
print("\n" + "="*60)
print("Target Variable: average_rating")
print("="*60)
print(df_raw['average_rating'].describe())
print(f"\nRating = 0.0 (suspicious): {(df_raw['average_rating'] == 0).sum()} books")
print(f"Rating = 5.0 (suspicious): {(df_raw['average_rating'] == 5).sum()} books")


Target Variable: average_rating
count    11119.000000
mean         3.934135
std          0.350384
min          0.000000
25%          3.770000
50%          3.960000
75%          4.135000
max          5.000000
Name: average_rating, dtype: float64

Rating = 0.0 (suspicious): 25 books
Rating = 5.0 (suspicious): 22 books


In [10]:
# Plot 1: Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Target Variable: average_rating", fontsize=13, fontweight='bold')

axes[0].hist(df_raw['average_rating'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution of average_rating")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].axvline(df_raw['average_rating'].mean(), color='crimson', linestyle='--', label=f"Mean = {df_raw['average_rating'].mean():.2f}")
axes[0].legend()

# Zoom in on the realistic range (most books cluster 3.0-4.5)
zoom = df_raw[df_raw['average_rating'] > 0]
axes[1].hist(zoom['average_rating'], bins=60, color='teal', edgecolor='white')
axes[1].set_title("Zoomed: ratings between 1.5 - 5")
axes[1].set_xlabel("Rating")
axes[1].axvline(zoom['average_rating'].mean(), color='crimson', linestyle='--', label=f"Mean = {zoom['average_rating'].mean():.2f}")
axes[1].legend()
plt.tight_layout()
plt.savefig('plot_01_target_distribution.png')
plt.close()
print("\n[Saved] plot_01_target_distribution.png")


[Saved] plot_01_target_distribution.png


# KEY INSIGHT
- 94% of all ratings sit between 3.0 and 4.5. 
- The distribution is left-skewed (few bad books survive long enough to accumulate ratings). 
- Thus, the model will be biased toward the 3.5-4.2 range and will struggle to predict extreme values (0, 1, 5).

In [ ]:
# 3.  Column-by-Column Useful Audit
print("\n" + "="*60)
print("3. COLUMN-BY-COLUMN USEFULNESS AUDIT")
print("="*60)

print("\n bookID: ")
corr_id = df_raw['bookID'].corr(df_raw['average_rating'])
print(f"  Unique values : {df_raw['bookID'].nunique()}")
print(f"  Range         : {df_raw['bookID'].min()} → {df_raw['bookID'].max()}")
print(f"  Correlation with rating: {corr_id:.5f}")
print("""
  VERDICT: DROP
  bookID is an arbitrary database key, it is not 1,2,3,4...
  it jumps (1,2,4,5,8,9...) and goes up to 45,641 for 11,119 books.
  Correlation with rating is essentially 0 (-0.035).
  If the model learns from bookID it will just memorise training rows,
  which is overfitting not learning any generalizable pattern.
""")


3. COLUMN-BY-COLUMN USEFULNESS AUDIT

 bookID: 
  Unique values : 11119
  Range         : 1 → 45641
  Correlation with rating: -0.03548

  VERDICT: DROP
  bookID is an arbitrary database key, it is not 1,2,3,4...
  it jumps (1,2,4,5,8,9...) and goes up to 45,641 for 11,119 books.
  Correlation with rating is essentially 0 (-0.035).
  If the model learns from bookID it will just memorise training rows,
  which is overfitting  not learning any generalizable pattern.



In [15]:
print("isbn / isbn13:")
corr_isbn13 = df_raw['isbn13'].corr(df_raw['average_rating'])
print(f"  isbn13 sample : {df_raw['isbn13'].head(3).tolist()}")
print(f"  isbn13 range  : {df_raw['isbn13'].min():.0f} → {df_raw['isbn13'].max():.0f}")
print(f"  Correlation with rating: {corr_isbn13:.6f}")
print("""
  VERDICT: DROP
  ISBN-13 starts with 978 (prefix for all books internationally).
  The remaining digits encode: publisher group, publisher, title, check digit.
  The number itself has no relationship to quality or rating.
  Correlation: -0.002 (pure noise). Using it would cause the model
  to memorize book identifiers, not learn patterns.

  NOTE: isbn (10-digit) is even less structured keep neither.
""")

isbn / isbn13:
  isbn13 sample : [9780439785969, 9780439358071, 9780439554893]
  isbn13 range  : 8987059752 → 9790007672386
  Correlation with rating: -0.001958

  VERDICT: DROP
  ISBN-13 starts with 978 (prefix for all books internationally).
  The remaining digits encode: publisher group, publisher, title, check digit.
  The number itself has no relationship to quality or rating.
  Correlation: -0.002 (pure noise). Using it would cause the model
  to memorize book identifiers, not learn patterns.

  NOTE: isbn (10-digit) is even less structured keep neither.



In [19]:
print("title:")
print(f"  Unique titles: {df_raw['title'].nunique()}")
print("""
  VERDICT: Engineer then DROP the raw string
  The raw title string is text, models need numbers.
  But the title contains useful information we can extract:
    1. is_series   : does the title contain "(Series #N)"?
    2. series_num  : what number in the series?
    3. title_length: word count (longer titles = academic/niche?)
  After extracting these features, drop the raw title column.
""")

title:
  Unique titles: 10344

  VERDICT: Engineer then DROP the raw string
  The raw title string is text, models need numbers.
  But the title contains useful information we can extract:
    1. is_series   : does the title contain "(Series #N)"?
    2. series_num  : what number in the series?
    3. title_length: word count (longer titles = academic/niche?)
  After extracting these features, drop the raw title column.



In [22]:
print("authors:")
print(f"  Unique author strings : {df_raw['authors'].nunique()}")
all_authors = df_raw['authors'].str.split('/').explode().str.strip()
print(f"  Unique individual authors: {all_authors.nunique()}")
print("""
  VERDICT: Engineer use num_authors + author_avg_rating
  9,227 unique authors. One-Hot Encoding would create 9,227 columns
  (memory explosion, curse of dimensionality, most are zeros).

  Better strategies:
    1. num_authors: count how many authors wrote the book (easy, numeric)
    2. author_avg_rating (target encoding): replace author with the
       average rating of their other books captures "author reputation"
       without creating thousands of columns.
       Must compute on training set and map to test set.
""")


authors:
  Unique author strings : 6635
  Unique individual authors: 9227

  VERDICT: Engineer use num_authors + author_avg_rating
  9,227 unique authors. One-Hot Encoding would create 9,227 columns
  (memory explosion, curse of dimensionality, most are zeros).

  Better strategies:
    1. num_authors: count how many authors wrote the book (easy, numeric)
    2. author_avg_rating (target encoding): replace author with the
       average rating of their other books captures "author reputation"
       without creating thousands of columns.
       Must compute on training set and map to test set.



In [24]:
print("language_code:")
lang_counts = df_raw['language_code'].value_counts()
print(lang_counts)
lang_stats = df_raw.groupby('language_code')['average_rating'].mean().sort_values(ascending=False)
print("\n  Avg rating per language:")
print(lang_stats)
print("""
  VERDICT: Keep. One-Hot Encode after grouping
  26 unique language codes, but 90%+ are eng/en-US/en-GB.
  Strategy:
    1. Merge English variants (en-US, en-GB, en-CA to eng)
    2. Keep only top 6 languages (eng, spa, fre, ger, jpn, zho)
    3. All others to 'other'
  Then OHE (One-Hot Encode) => 7 binary columns.
  Small but real difference in avg rating by language.
""")

language_code:
language_code
eng      8906
en-US    1406
spa       218
en-GB     214
fre       144
ger        99
jpn        46
mul        19
zho        14
grc        11
por        10
en-CA       7
ita         5
enm         3
lat         3
rus         2
swe         2
ara         1
nl          1
srp         1
msa         1
glg         1
wel         1
nor         1
tur         1
gla         1
ale         1
Name: count, dtype: int64

  Avg rating per language:
language_code
wel      5.000000
gla      4.470000
zho      4.456429
tur      4.420000
ale      4.360000
lat      4.353333
jpn      4.268696
rus      4.255000
nl       4.180000
mul      4.126316
msa      4.110000
ita      4.078000
en-CA    4.025714
fre      3.971528
ger      3.950101
por      3.945000
eng      3.934078
spa      3.929312
en-GB    3.923411
en-US    3.915000
enm      3.873333
grc      3.707273
nor      3.600000
ara      3.550000
swe      3.455000
glg      3.360000
srp      0.000000
Name: average_rating, dtype: float64

 

In [27]:
print("num_pages:")
corr_pages = df_raw['num_pages'].corr(df_raw['average_rating'])
print(f"  Range  : {df_raw['num_pages'].min()} => {df_raw['num_pages'].max()}")
print(f"  Zeros  : {(df_raw['num_pages'] == 0).sum()} rows")
print(f"  Correlation with rating: {corr_pages:.4f}")
print("""
  VERDICT: Keep, clean zeros first
  Best linear correlation of any single raw numeric feature (+0.15).
  Why does page count predict rating? Selection bias:
  Readers only finish (and rate) long books if they genuinely liked them.
  76 rows have 0 pages => imputation or removal needed (see Step 2).
""")

num_pages:
  Range  : 0 => 6576
  Zeros  : 76 rows
  Correlation with rating: 0.1507

  VERDICT: Keep, clean zeros first
  Best linear correlation of any single raw numeric feature (+0.15).
  Why does page count predict rating? Selection bias:
  Readers only finish (and rate) long books if they genuinely liked them.
  76 rows have 0 pages => imputation or removal needed (see Step 2).



In [ ]:
print("ratings_count:")
corr_rc = df_raw['ratings_count'].corr(df_raw['average_rating'])
log_rc = np.log1p(df_raw['ratings_count'])
corr_log_rc = log_rc.corr(df_raw['average_rating'])
print(f"  Range  : {df_raw['ratings_count'].min()} → {df_raw['ratings_count'].max():,}")
print(f"  Zeros  : {(df_raw['ratings_count'] == 0).sum()} rows")
print(f"  Correlation (raw)      : {corr_rc:.4f}")
print(f"  Correlation (log1p)    : {corr_log_rc:.4f}")
print("""
  VERDICT: Keep but log-transform
  Extreme skew (0 to 4.5M). After log1p(), correlation doubles (+0.14).
  Why: popular books tend to be both good AND heavily rated (positive loop).
  80 rows with ratings_count = 0 > these books have an unreliable rating.
""")

ratings_count:
  Range  : 0 → 4,597,666
  Zeros  : 80 rows
  Correlation (raw)      : 0.0382
  Correlation (log1p)    : 0.1428

  VERDICT: Keep but log-transform
  Extreme skew (0 to 4.5M). After log1p(), correlation doubles (+0.14).
  Why: popular books tend to be both good AND heavily rated (positive loop).
  80 rows with ratings_count = 0 > these books have an unreliable rating.



In [32]:
print("text_reviews_count:")
corr_tr = df_raw['text_reviews_count'].corr(df_raw['average_rating'])
log_tr = np.log1p(df_raw['text_reviews_count'])
corr_log_tr = log_tr.corr(df_raw['average_rating'])
print(f"  Correlation (raw)   : {corr_tr:.4f}")
print(f"  Correlation (log1p) : {corr_log_tr:.4f}")
print("""
  VERDICT: Keep but log-transform
  Correlated with ratings_count (same popularity effect).
  Adds independent signal: ratio of text_reviews/ratings tells us
  how "discussable" a book is, controversial books get debated more.
""")

text_reviews_count:
  Correlation (raw)   : 0.0336
  Correlation (log1p) : 0.0796

  VERDICT: Keep but log-transform
  Correlated with ratings_count (same popularity effect).
  Adds independent signal: ratio of text_reviews/ratings tells us
  how "discussable" a book is, controversial books get debated more.



In [37]:
print("publication_date:")
df_raw['pub_date_parsed'] = pd.to_datetime(df_raw['publication_date'], errors='coerce')
print(f"  Null after parsing: {df_raw['pub_date_parsed'].isna().sum()}")
print(f"  Year range: {df_raw['pub_date_parsed'].dt.year.min():.0f} => {df_raw['pub_date_parsed'].dt.year.max():.0f}")
pub_year = df_raw['pub_date_parsed'].dt.year
corr_year = pub_year.corr(df_raw['average_rating'])
print(f"  Correlation (year): {corr_year:.4f}")
print("""
  VERDICT: Extract year Keep as pub_year (weak signal)
  The raw string (e.g. "9/16/2006") must be parsed.
  Only the year matters month/day adds noise, not signal.
  Weak negative correlation (-0.03): older books have marginally
  higher ratings (classics survive; bad old books are forgotten).
  Drop the original string column after extraction.
""")

publication_date:
  Null after parsing: 2
  Year range: 1900 => 2020
  Correlation (year): -0.0318

  VERDICT: Extract year Keep as pub_year (weak signal)
  The raw string (e.g. "9/16/2006") must be parsed.
  Only the year matters month/day adds noise, not signal.
  Weak negative correlation (-0.03): older books have marginally
  higher ratings (classics survive; bad old books are forgotten).
  Drop the original string column after extraction.



In [39]:
print("publisher:")
print(f"  Unique publishers : {df_raw['publisher'].nunique()}")
pub_mean = df_raw.groupby('publisher')['average_rating'].mean()
df_raw['publisher_encoded'] = df_raw['publisher'].map(pub_mean)
corr_pub = df_raw['publisher_encoded'].corr(df_raw['average_rating'])
print(f"  Correlation via target encoding (TRAIN ONLY): {corr_pub:.4f}")
print("""
  VERDICT: Keep use target encoding (strongest signal!)
  2,289 unique publishers OHE is impossible.
  Target encoding: replace publisher name with the mean rating
  of all books from that publisher IN THE TRAINING SET.
  Result: 1 numeric column. Strongest signal in the dataset (0.68).
  CRITICAL: compute encoding on training data only, then map to test.
      Computing it on all data would be data leakage.

  Also: 2 rows have a date as publisher ("10/18") a CSV shift error.
  These must be corrected or dropped in cleaning.
""")

publisher:
  Unique publishers : 2289
  Correlation via target encoding (TRAIN ONLY): 0.6836

  VERDICT: Keep use target encoding (strongest signal!)
  2,289 unique publishers OHE is impossible.
  Target encoding: replace publisher name with the mean rating
  of all books from that publisher IN THE TRAINING SET.
  Result: 1 numeric column. Strongest signal in the dataset (0.68).
  CRITICAL: compute encoding on training data only, then map to test.
      Computing it on all data would be data leakage.

  Also: 2 rows have a date as publisher ("10/18") a CSV shift error.
  These must be corrected or dropped in cleaning.



In [43]:
# 4.  Series Relationship
print("\n" + "="*60)
print("Series Relationship")
print("="*60)

df_raw['series_num'] = df_raw['title'].str.extract(r'\(.*?#(\d+)\)').astype(float)
df_raw['is_series'] = df_raw['series_num'].notna().astype(int)
series_mask = df_raw['series_num'].notna()
print(f"  Books with a '#N' series marker in title: {series_mask.sum()}")
print(f"  Correlation (series_num vs rating): {df_raw['series_num'].corr(df_raw['average_rating']):.4f}")
print(f"  Correlation (is_series vs rating) : {df_raw['is_series'].corr(df_raw['average_rating']):.4f}")
print()

series_stats = df_raw.groupby('is_series')['average_rating'].agg(['mean', 'std', 'count'])
series_stats.index = ['Standalone', 'Series']
print(series_stats)
print("""
  VERDICT: Partial extract is_series + series_num as features
  Intuition: "HarryPorter(HP)#6 came after HP#5 that's useful context."
  The idea is sound in spirit but has a fundamental problem:

  A regression model sees each book as an independent row.
  It doesn't link "HP#6" to "HP#5" automatically.
  So to perform this relationship, we need:
    - A time-series model (LSTM, etc.) far beyond the project scope
    - Or a graph model linking books also far beyond scope

  What we can do:
    1. is_series (0/1): series books rate +0.05 points higher on average
    2. series_num (#1, #2, #3...): weak negative correlation (-0.07)
       meaning later entries in a series rate slightly lower
       (fans who stayed may still rate well, but general audience shrinks)

  These two simple features are useful just not in the deeper way.
""")


Series Relationship
  Books with a '#N' series marker in title: 2246
  Correlation (series_num vs rating): -0.0697
  Correlation (is_series vs rating) : 0.0568

                mean       std  count
Standalone  3.924129  0.371079   8873
Series      3.973664  0.248719   2246

  VERDICT: Partial extract is_series + series_num as features
  Intuition: "HarryPorter(HP)#6 came after HP#5 that's useful context."
  The idea is sound in spirit but has a fundamental problem:

  A regression model sees each book as an independent row.
  It doesn't link "HP#6" to "HP#5" automatically.
  So to perform this relationship, we need:
    - A time-series model (LSTM, etc.) far beyond the project scope
    - Or a graph model linking books also far beyond scope

  What we can do:
    1. is_series (0/1): series books rate +0.05 points higher on average
    2. series_num (#1, #2, #3...): weak negative correlation (-0.07)
       meaning later entries in a series rate slightly lower
       (fans who stayed m

In [49]:
# 5. Data Quality Problems
print("\n" + "="*60)
print("5. DATA QUALITY PROBLEMS")
print("="*60)

# Problem 1: Bad CSV rows (parsing errors)
print("Problem 1: Malformed CSV rows")
print("  ~8 lines have too many commas (unquoted commas in titles/authors).")
print("  These are silently skipped with on_bad_lines='skip'.")
print("  Action: SKIP (they are a tiny fraction < 0.1% of data).")
print()

# Problem 2: Publisher = date
bad_pub = df_raw[df_raw['publisher'].str.match(r'^\d{1,2}/\d{2}$', na=False)]
print(f"Problem 2: Publisher column contains a date value: {len(bad_pub)} rows")
print(bad_pub[['title', 'publisher', 'publication_date']])
print("  Root cause: '10/18' is a French publisher code (short for 'Union Générale d'Éditions 10/18').")
print("  This is not a data error, '10/18' is the real publisher name.")
print("  Action: Keep these rows. The filter we had applied was wrong here.")
print()

# Problem 3: 0-page books
zero_pages = df_raw[df_raw['num_pages'] == 0]
print(f"Problem 3: Books with 0 pages: {len(zero_pages)} rows")
print(zero_pages[['title', 'num_pages', 'ratings_count']].head(5))
print("  Action: Can't be 0 pages, these are missing values stored as 0.")
print("  Strategy: impute with median pages from the same publisher or global median.")
print()

# Problem 4: 0 ratings with a non-zero rating
zero_rc_nonzero_rating = df_raw[(df_raw['ratings_count'] == 0) & (df_raw['average_rating'] > 0)]
print(f"Problem 4: Books with 0 ratings but a non-zero average_rating: {len(zero_rc_nonzero_rating)}")
print(zero_rc_nonzero_rating[['title', 'ratings_count', 'average_rating']].head(5))
print("  This is logically impossible (can't have an average with 0 values).")
print("  Action: DROP these rows the rating is untrustworthy.")
print()

# Problem 5: rating = 0 with 0 ratings
zero_both = df_raw[(df_raw['ratings_count'] == 0) & (df_raw['average_rating'] == 0)]
print(f"Problem 5: Books with rating=0 AND ratings=0 (unpublished/no data): {len(zero_both)}")
print("  Action: DROP these books haven't been rated. Our target is meaningless for them.")
print()

# Problem 6: Duplicate books (same title + same author)
dupes = df_raw[df_raw.duplicated(subset=['title', 'authors'], keep=False)]
print(f"Problem 6: Duplicate (title + author) rows: {len(dupes)} rows involved")
print("  Root cause: same book published by different publishers OR different editions.")
print("  These are NOT errors they are valid separate editions. KEEP them.")
print("  The model can learn that the same book, published by a prestige publisher,")
print("  might have a marginally different profile.")
print()


5. DATA QUALITY PROBLEMS
Problem 1: Malformed CSV rows
  ~8 lines have too many commas (unquoted commas in titles/authors).
  These are silently skipped with on_bad_lines='skip'.
  Action: SKIP (they are a tiny fraction < 0.1% of data).

Problem 2: Publisher column contains a date value: 2 rows
             title publisher publication_date
6898     Glamorama     10/18        2/15/2001
7323  La mezzanine     10/18       11/18/1998
  Root cause: '10/18' is a French publisher code (short for 'Union Générale d'Éditions 10/18').
  This is not a data error, '10/18' is the real publisher name.
  Action: Keep these rows. The filter we had applied was wrong here.

Problem 3: Books with 0 pages: 76 rows
                                                  title  num_pages  \
306   The 5 Love Languages / The 5 Love Languages Jo...          0   
853                    The Tragedy of Pudd'nhead Wilson          0   
1061  Murder by Moonlight & Other Mysteries (New Adv...          0   
1064  The Unfort

In [50]:
# 6.  Correlations Visual Summary
df_raw['log_ratings'] = np.log1p(df_raw['ratings_count'])
df_raw['log_reviews'] = np.log1p(df_raw['text_reviews_count'])
df_raw['pub_year'] = df_raw['pub_date_parsed'].dt.year
df_raw['num_authors'] = df_raw['authors'].str.split('/').apply(len)
df_raw['reviews_per_rating'] = np.where(
    df_raw['ratings_count'] > 0,
    df_raw['text_reviews_count'] / df_raw['ratings_count'], 0
)

numeric_features = [
    'num_pages', 'log_ratings', 'log_reviews',
    'pub_year', 'num_authors', 'reviews_per_rating',
    'series_num', 'is_series', 'average_rating'
]
corr_matrix = df_raw[numeric_features].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Feature Correlations", fontsize=13, fontweight='bold')

# Heatmap
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, ax=axes[0], linewidths=0.5,
    annot_kws={"size": 8}
)
axes[0].set_title("Correlation Matrix (all features)")
axes[0].tick_params(axis='x', rotation=45)

# Bar chart: correlation with target only
target_corr = corr_matrix['average_rating'].drop('average_rating').sort_values()
colors = ['crimson' if v < 0 else 'steelblue' for v in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title("Correlation with average_rating")
axes[1].set_xlabel("Pearson r")
for i, (v, name) in enumerate(zip(target_corr.values, target_corr.index)):
    axes[1].text(v + 0.003 if v >= 0 else v - 0.003, i,
                 f"{v:.3f}", va='center',
                 ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('plot_02_correlations.png')
plt.close()
print("[Saved] plot_02_correlations.png")

[Saved] plot_02_correlations.png


In [51]:
# 7.  Outlier Inspection
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Outlier Inspection", fontsize=13, fontweight='bold')

for ax, col in zip(axes, ['num_pages', 'ratings_count', 'text_reviews_count']):
    clean = df_raw[df_raw[col] > 0][col]
    ax.boxplot(clean, vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightsteelblue'))
    ax.set_title(col)
    ax.set_ylabel("Value")
    q99 = clean.quantile(0.99)
    ax.annotate(f"99th pct: {q99:,.0f}", xy=(1, q99),
                xytext=(1.15, q99), fontsize=8)

plt.tight_layout()
plt.savefig('plot_03_outliers.png')
plt.close()
print("[Saved] plot_03_outliers.png")

[Saved] plot_03_outliers.png


In [52]:
# 8.  Language & Publisher Signal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Categorical Feature Signals", fontsize=13, fontweight='bold')

# Language
lang_map = {'en-US': 'eng', 'en-GB': 'eng', 'en-CA': 'eng'}
df_plot = df_raw.copy()
df_plot['lang_grouped'] = df_plot['language_code'].replace(lang_map)
top_langs = df_plot['lang_grouped'].value_counts().nlargest(6).index
df_plot['lang_grouped'] = df_plot['lang_grouped'].apply(
    lambda x: x if x in top_langs else 'other')
lang_order = df_plot.groupby('lang_grouped')['average_rating'].median().sort_values(ascending=False).index
sns.boxplot(data=df_plot, x='lang_grouped', y='average_rating',
            order=lang_order, ax=axes[0], palette='Set2')
axes[0].set_title("Rating by Language (grouped)")
axes[0].set_xlabel("language_code")
axes[0].tick_params(axis='x', rotation=30)

# Publisher (top 15 by book count)
top_pubs = df_raw['publisher'].value_counts().nlargest(15).index
pub_df = df_raw[df_raw['publisher'].isin(top_pubs)]
pub_order = pub_df.groupby('publisher')['average_rating'].median().sort_values(ascending=False).index
sns.boxplot(data=pub_df, x='publisher', y='average_rating',
            order=pub_order, ax=axes[1], palette='Set3')
axes[1].set_title("Rating by Publisher (Top 15 by volume)")
axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('plot_04_categorical_signals.png')
plt.close()
print("[Saved] plot_04_categorical_signals.png")

[Saved] plot_04_categorical_signals.png


In [53]:
# 9.  Final Feature Decision Table
print("\n" + "="*60)
print("9. FINAL FEATURE DECISION TABLE")
print("="*60)
print("""
╔══════════════════════╦══════╦═══════════════════════════════════════════════════════╗
║ Column               ║ Keep ║ How to use it                                         ║
╠══════════════════════╬══════╬═══════════════════════════════════════════════════════╣
║ bookID               ║  NO  ║ Arbitrary key, DROP                                   ║
║ title                ║  ~~  ║ Extract: is_series, series_num, title_word_count       ║
║ authors              ║  ~~  ║ Extract: num_authors, author_avg_rating (target enc.)  ║
║ average_rating       ║  TAR  ║ TARGET VARIABLE                                        ║
║ isbn                 ║  NO  ║ Identifier, DROP                                      ║
║ isbn13               ║  NO  ║ Identifier, corr=-0.002, DROP                         ║
║ language_code        ║  YES  ║ Group English variants, OHE (7 categories)            ║
║ num_pages            ║  YES  ║ Numeric, clean zeros > impute                          ║
║ ratings_count        ║  YES  ║ log1p() transform > strong signal                      ║
║ text_reviews_count   ║  YES  ║ log1p() transform                                      ║
║ publication_date     ║  ~~  ║ Parse > extract pub_year only > DROP original          ║
║ publisher            ║  YES  ║ Target encoding (mean rating per publisher)             ║
╚══════════════════════╩══════╩═══════════════════════════════════════════════════════╝

ENGINEERED FEATURES TO ADD:
  • is_series          (0/1): does the title contain a series number?
  • series_num         (int): which number in the series (#1, #2, #3...)
  • num_authors        (int): how many authors wrote this book
  • author_avg_rating  (float): target encoding per author (TRAIN SET ONLY)
  • reviews_per_rating (float): text_reviews / ratings_count (book 'debatability')
  • pub_year           (int): extracted from publication_date

DATA PROBLEMS TO CLEAN (Step 2):
  • 76 rows with num_pages = 0  => impute with global or publisher median
  • 25 rows with rating = 0     => DROP (no real rating exists)
  • 80 rows with ratings = 0    => DROP (rating is statistically unreliable)
  • ~8 malformed CSV rows       => already handled at load time
  • '10/18' publisher           => this IS a real publisher name, KEEP it
""")

print("EDA complete. Plots saved:")
print("  plot_01_target_distribution.png")
print("  plot_02_correlations.png")
print("  plot_03_outliers.png")
print("  plot_04_categorical_signals.png")


9. FINAL FEATURE DECISION TABLE

╔══════════════════════╦══════╦═══════════════════════════════════════════════════════╗
║ Column               ║ Keep ║ How to use it                                         ║
╠══════════════════════╬══════╬═══════════════════════════════════════════════════════╣
║ bookID               ║  NO  ║ Arbitrary key, DROP                                   ║
║ title                ║  ~~  ║ Extract: is_series, series_num, title_word_count       ║
║ authors              ║  ~~  ║ Extract: num_authors, author_avg_rating (target enc.)  ║
║ average_rating       ║  TAR  ║ TARGET VARIABLE                                        ║
║ isbn                 ║  NO  ║ Identifier, DROP                                      ║
║ isbn13               ║  NO  ║ Identifier, corr=-0.002, DROP                         ║
║ language_code        ║  YES  ║ Group English variants, OHE (7 categories)            ║
║ num_pages            ║  YES  ║ Numeric, clean zeros > impute                   